In [1]:
import os
import warnings
import seaborn as sns
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.model_selection import LeaveOneGroupOut, ParameterGrid
from sklearn.metrics import f1_score, balanced_accuracy_score, recall_score, precision_score, confusion_matrix
from xgboost import XGBClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [2]:
# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
seed = 69
torch.manual_seed(seed)
np.random.seed(seed)

# Define root directory
root = '.'

In [3]:
df = pd.read_csv('./new_dataset/maison-llf-features.CSV', sep=",")

In [4]:
count_vec = CountVectorizer()

In [5]:
ana = pd.read_csv('./new_dataset/maison-llf-demographics.CSV', sep=",")

In [6]:
ana_col = list(ana.columns)

In [7]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = count_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"CountVect_{w}" for w in count_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [8]:
tfidf_vec = TfidfVectorizer()

In [9]:
for column in ana_col:

    if column != 'age' and column != 'participant':

        #print(column)
        
        count_matrix = tfidf_vec.fit_transform(ana[column])
        count_df = pd.DataFrame(
            count_matrix.toarray(),
            columns=[f"TfidVect_{w}" for w in tfidf_vec.get_feature_names_out()]
        )
        ana = pd.concat([ana, count_df], axis=1)

In [10]:
num_ana = ana.drop(["sex", "fracture-type", "relationship", "education", "work", "ethnicity"], axis=1)

In [11]:
num_ana.head(5)

,participant,age,CountVect_female,CountVect_male,CountVect_femur,CountVect_fracture,CountVect_hip,CountVect_pelvis,CountVect_replacement,CountVect_divorced,...,TfidVect_secondary,TfidVect_undergraduate,TfidVect_employed,TfidVect_part,TfidVect_retired,TfidVect_time,TfidVect_asian,TfidVect_black,TfidVect_caucasian,TfidVect_south
0,1,74,1,0,0,1,1,0,0,0,...,0.000000,0.782749,0.00000,0.00000,1.0,0.00000,0.0,1.0,0.0,0.0
1,2,66,1,0,0,1,1,0,0,0,...,0.000000,0.782749,0.57735,0.57735,0.0,0.57735,0.0,0.0,1.0,0.0
2,3,70,1,0,0,1,0,1,0,0,...,0.707107,0.000000,0.00000,0.00000,1.0,0.00000,0.0,0.0,1.0,0.0
3,4,60,1,0,1,1,0,0,0,1,...,0.707107,0.000000,0.00000,0.00000,1.0,0.00000,0.0,0.0,1.0,0.0
4,5,75,0,1,0,0,1,0,1,0,...,0.000000,0.000000,0.57735,0.57735,0.0,0.57735,0.0,0.0,1.0,0.0


In [12]:
data = df.merge(num_ana, how='right', on='participant')

In [13]:
# Convert tensors to pandas DataFrame
df = data[["sis", "ohs", "oks"]]

In [14]:
# Compute quartiles for discretization
siss_q1, siss_q2, siss_q3 = np.percentile(df["sis"], [25, 50, 75])
ohss_q1, ohss_q2, ohss_q3 = np.percentile(df["ohs"], [25, 50, 75])
okss_q1, okss_q2, okss_q3 = np.percentile(df["oks"], [25, 50, 75])

In [15]:
# Define quartile-based bins and labels
quartile_labels = [0, 1, 2, 3]

In [16]:
# Apply discretization
df["SISS_Category_Q"] = pd.cut(df["sis"], bins=[df["sis"].min(), siss_q1, siss_q2, siss_q3, df["sis"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)
df["OHSS_Category_Q"] = pd.cut(df["ohs"], bins=[df["ohs"].min(), ohss_q1, ohss_q2, ohss_q3, df["ohs"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

df["OKSS_Category_Q"] = pd.cut(df["oks"], bins=[df["oks"].min(), okss_q1, okss_q2, okss_q3, df["oks"].max()],
                               labels=quartile_labels, include_lowest=True).astype(int)

In [18]:
resp = "OHSS_Category_Q"
y = df[resp]

In [19]:
# Extract features, target, and patient IDs for LOPO
X = data.iloc[:, -104:-3]
groups = data["participant"]

In [20]:
#84-39 + 1

In [21]:
#X.info()

In [22]:
# Define classifier and hyperparameter grid
#model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
param_grid = {
            "n_estimators": [20, 50, 100],
            "max_depth": [3, 6, 8],
            "learning_rate": [0.001, 0.01, 0.1, 0.3]
            #"subsample": [0.8, 1.0],
            #"colsample_bytree": [0.8, 1.0],
            #"reg_lambda": [1.0, 5.0],
        }

In [23]:
# Leave-One-Patient-Out CV (LOPO)
outer_logo = LeaveOneGroupOut()

In [24]:
# Initialize results storage
overall_conf_matrix = np.zeros((4, 4))  # Assuming 4 categories (0, 1, 2, 3)
performance_metrics = []

In [26]:
# Outer LOPO Loop
count=0

for train_idx, test_idx in outer_logo.split(X, y, groups):
    #print(train_idx) index
    count=count+1
    print(count)
    
    #if count == 6 or count == 7:
    #    continue 
    X_train_outer, X_test = X.iloc[train_idx].to_numpy(), X.iloc[test_idx].to_numpy()
    y_train_outer, y_test = y.iloc[train_idx].to_numpy(), y.iloc[test_idx].to_numpy()
    groups_train_outer = groups.iloc[train_idx]
    print(groups_train_outer.unique())
    
    # Inner LOPO for Hyperparameter Optimization
    inner_logo = LeaveOneGroupOut()
    best_model = None
    best_score = -np.inf
    
    for inner_train_idx, inner_val_idx in inner_logo.split(X_train_outer, y_train_outer, groups_train_outer):
        X_train_inner, X_val = X_train_outer[inner_train_idx], X_train_outer[inner_val_idx]
        y_train_inner, y_val = y_train_outer[inner_train_idx], y_train_outer[inner_val_idx]
        groups_train_inner = groups_train_outer.iloc[inner_train_idx]
        print(groups_train_inner.unique())
        
        # Hyperparameter tuning
        for params in ParameterGrid(param_grid):
            params = {k: int(v) if isinstance(v, np.generic) else v for k, v in params.items()}
            model = XGBClassifier(use_label_encoder=False, eval_metric="mlogloss")
            print(params)
            model.set_params(**params)
            model.fit(X_train_inner, y_train_inner)
            y_val_pred = model.predict(X_val)
            score = f1_score(y_val, y_val_pred, average="macro")
            # https://scikit-learn.org/stable/modules/generated/sklearn.metrics.f1_score.html

            if score > best_score:
                best_score = score
                best_model = model
                best_params = params  # Store the best hyperparameters

    # Train best model on full outer training set
    best_model.fit(X_train_outer, y_train_outer)
    y_pred = best_model.predict(X_test)

    # Compute metrics
    f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred, average="macro")
    precision = precision_score(y_test, y_pred, average="macro")
    conf_matrix = confusion_matrix(y_test, y_pred, labels=[0, 1, 2, 3])

    # Aggregate confusion matrices
    overall_conf_matrix += conf_matrix

    # Store results
    performance_metrics.append([f1, balanced_acc, recall, precision])

1
[ 2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
[ 3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18]
{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 20}
{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 50}
{'learning_rate': 0.001, 'max_depth': 3, 'n_estimators': 100}
{'learning_rate': 0.001, 'max_depth': 6, 'n_estimators': 20}
{'learning_rate': 0.001, 'max_depth': 6, 'n_estimators': 50}
{'learning_rate': 0.001, 'max_depth': 6, 'n_estimators': 100}
{'learning_rate': 0.001, 'max_depth': 8, 'n_estimators': 20}
{'learning_rate': 0.001, 'max_depth': 8, 'n_estimators': 50}
{'learning_rate': 0.001, 'max_depth': 8, 'n_estimators': 100}
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 20}
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 50}
{'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 100}
{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 20}
{'learning_rate': 0.01, 'max_depth': 6, 'n_estimators': 50}
{'learning_rate': 0.01, 'max_depth': 6, 'n

KeyboardInterrupt: 

[ 1  3  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  4  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  5  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  6  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  7  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  8  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  9 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8 11 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 12 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 13 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 14 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 15 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 16 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 17 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 16 18]
[ 1  2  3  4  5  6  7  8  9 11 12 13 14 15 16 17]
11
[ 1  2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 2  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 1  3  4  5  6  7  8  9 10 12 13 14 15 16 17 18]
[ 1  2  4  5  6  7  8  9 10 12 13 14 15 16 1

In [ ]:
overall_conf_matrix

In [ ]:
# Convert results to DataFrame
performance_df = pd.DataFrame(performance_metrics, columns=["Macro-F1", "Balanced Accuracy", "Macro Recall", "Macro Precision"])

In [ ]:
performance_df

In [ ]:
assert False

In [ ]:
# Save performance metrics and overall confusion matrix
output_path = os.path.join(root, "results/model_performance_ohss.xlsx")
with pd.ExcelWriter(output_path) as writer:
    performance_df.to_excel(writer, sheet_name="All_Folds")

In [ ]:
# Plot overall confusion matrix
plt.figure(figsize=(6, 5))
sns.heatmap(overall_conf_matrix, annot=True, cmap="coolwarm", xticklabels=[0, 1, 2, 3], yticklabels=[0, 1, 2, 3])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Overall Confusion Matrix OHSS")
conf_matrix_path = os.path.join(root, "results/overall_confusion_matrix_ohss.pdf")
plt.savefig(conf_matrix_path, dpi=300, bbox_inches='tight')
plt.close()

print(f"\n✅ Performance metrics saved as: {output_path}")
print(f"✅ Overall Confusion Matrix saved as: {conf_matrix_path}")